In [2]:
# -- Cell 1: Install dependencies ------------------------------------------
# Run once per session / Executar uma vez por sessão

# ============================================================
# CÉLULA 1 — Instalar dependências
# ============================================================

%pip install sentence-transformers transformers torch datasets scikit-learn scipy pandas -q
print("\u2713 Dependências instaladas")

Note: you may need to restart the kernel to use updated packages.
✓ Dependências instaladas


In [1]:
# -- Cell 2: Synthetic training pairs (pathway, NANDA-I diagnosis, label) ---
# Pairs generated from actual STRING enrichment pathways (Up: 9, Down: 26)
# and NANDA-I diagnoses from clinicalbert_vias_AVC.ipynb (Cell 4).
#
# Label scale:
#   >= 0.85  high relevance   (direct immune–nursing alignment)
#   0.50–0.84 moderate relevance (indirect / secondary clinical link)
#   <= 0.20  hard negative     (immune pathway x non-immune functional diagnosis)
#
# ============================================================
# CÉLULA 2 — Pares de treinamento sintéticos
# Base: 65 pares; expansão abaixo para ≥150 (hard negatives + moderados)
# ============================================================

base_training_pairs = [
    ("Neutrophil degranulation",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.92),
    ("Innate Immune System",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.91),
    ("Innate immunity",
     "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
     0.88),
    ("Immune effector process",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.87),
    ("Immune response",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.86),
    ("Secretory granule",
     "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
     0.85),
    ("Neutrophil degranulation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.9),
    ("Innate Immune System",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.89),
    ("Immune effector process",
     "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
     0.86),
    ("Innate immunity",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.9),
    ("Lymphocyte activation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.88),
    ("Adaptive immune response",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.9),
    ("Unusual infection",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.93),
    ("B cell activation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.86),
    ("T cell differentiation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.87),
    ("Immunoglobulin production",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.88),
    ("Lymphocyte differentiation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.87),
    ("Adaptive immunity",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.91),
    ("Positive regulation of immune system process",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.87),
    ("Antigen receptor-mediated signaling pathway",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.86),
    ("Regulation of immune response",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.86),
    ("Immunoglobulin production",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.85),
    ("Neutrophil degranulation",
     "Acute confusion: abrupt onset of reversible disturbances of consciousness attention cognition and perception related to neuroinflammation and cerebral ischemia (NANDA-I 00128)",
     0.72),
    ("Innate immunity",
     "Risk for ineffective cerebral tissue perfusion: susceptibility to decrease in cerebral tissue circulation related to cerebral ischemia edema and blood-brain barrier disruption (NANDA-I 00201)",
     0.75),
    ("Immune response",
     "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",
     0.7),
    ("Secretory granule",
     "Acute pain: unpleasant sensory and emotional experience associated with actual or potential tissue damage related to neurological injury and post-stroke pain (NANDA-I 00132)",
     0.62),
    ("Innate Immune System",
     "Risk for injury: susceptibility to physical damage due to sensory and motor deficit impaired balance and altered consciousness after stroke (NANDA-I 00035)",
     0.58),
    ("Extracellular region",
     "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",
     0.67),
    ("Cytoplasmic vesicle",
     "Risk for pressure injury: susceptibility to localized damage to skin and underlying tissue usually over bony prominence related to immobility and sensory deficit (NANDA-I 00249)",
     0.56),
    ("Lymphocyte activation",
     "Anxiety: vague uneasy feeling of discomfort or dread accompanied by autonomic response related to neurological deficit uncertainty and loss of independence (NANDA-I 00146)",
     0.55),
    ("Adaptive immune response",
     "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",
     0.65),
    ("Regulation of immune response",
     "Decreased intracranial adaptive capacity: intracranial fluid dynamic mechanisms compromised resulting in repeated disproportionate increases in intracranial pressure (NANDA-I 00049)",
     0.64),
    ("Positive regulation of immune system process",
     "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
     0.78),
    ("Unusual infection",
     "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",
     0.7),
    ("CD22 mediated BCR regulation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.72),
    ("Immunological synapse formation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.74),
    ("Cell surface receptor signaling pathway",
     "Decreased intracranial adaptive capacity: intracranial fluid dynamic mechanisms compromised resulting in repeated disproportionate increases in intracranial pressure (NANDA-I 00049)",
     0.6),
    ("Positive regulation of T cell activation",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.79),
    ("Immune response",
     "Risk for ineffective cerebral tissue perfusion: susceptibility to decrease in cerebral tissue circulation related to cerebral ischemia edema and blood-brain barrier disruption (NANDA-I 00201)",
     0.68),
    ("Innate immunity",
     "Acute confusion: abrupt onset of reversible disturbances of consciousness attention cognition and perception related to neuroinflammation and cerebral ischemia (NANDA-I 00128)",
     0.72),
    ("Secretory granule lumen",
     "Risk for infection: susceptibility to invasion and multiplication of pathogenic organisms related to immunosuppression immobility and invasive procedures (NANDA-I 00004)",
     0.8),
    ("Neutrophil degranulation",
     "Risk for ineffective cerebral tissue perfusion: susceptibility to decrease in cerebral tissue circulation related to cerebral ischemia edema and blood-brain barrier disruption (NANDA-I 00201)",
     0.75),
    ("B cell proliferation",
     "Fatigue: overwhelming sustained sense of exhaustion and decreased capacity for physical and mental work related to neurological disease (NANDA-I 00093)",
     0.52),
    ("Immunoglobulin production",
     "Ineffective protection: decrease in ability to guard self from internal or external threats related to altered immune response and impaired adaptive immunity (NANDA-I 00043)",
     0.82),
    ("Neutrophil degranulation",
     "Disturbed sleep pattern: time-limited disruption of sleep amount and quality related to hospitalization neurological symptoms and pain (NANDA-I 00198)",
     0.05),
    ("Innate immunity",
     "Impaired physical mobility: limitation in independent purposeful physical movement related to neurological impairment and hemiplegia (NANDA-I 00085)",
     0.08),
    ("B cell receptor complex",
     "Bathing self-care deficit: impaired ability to independently perform bathing and hygiene activities related to hemiplegia and motor deficit (NANDA-I 00108)",
     0.05),
    ("Lymphocyte differentiation",
     "Constipation: decrease in normal bowel movement frequency and consistency related to immobility and neurological dysfunction (NANDA-I 00011)",
     0.06),
    ("Adaptive immune response",
     "Impaired verbal communication: decreased ability to receive process transmit and use a system of symbols related to aphasia after stroke (NANDA-I 00051)",
     0.07),
    ("T cell differentiation",
     "Dressing self-care deficit: impaired ability to dress and undress independently related to motor and sensory deficit after stroke (NANDA-I 00109)",
     0.05),
    ("Immunoglobulin production",
     "Interrupted family processes: change in family relationships and functioning related to acute illness and hospitalization of family member (NANDA-I 00060)",
     0.06),
    ("B cell activation",
     "Impaired physical mobility: limitation in independent purposeful physical movement related to neurological impairment and hemiplegia (NANDA-I 00085)",
     0.07),
    ("CD22 mediated BCR regulation",
     "Fatigue: overwhelming sustained sense of exhaustion and decreased capacity for physical and mental work related to neurological disease (NANDA-I 00093)",
     0.07),
    ("Immunological synapse formation",
     "Hopelessness: subjective state in which individual sees limited or no alternatives or personal choices and is unable to mobilize energy related to neurological disability (NANDA-I 00124)",
     0.05),
    ("B-cell antigen receptor complex-associated protein alpha/beta chain",
     "Constipation: decrease in normal bowel movement frequency and consistency related to immobility and neurological dysfunction (NANDA-I 00011)",
     0.04),
    ("Antigen receptor-mediated signaling pathway",
     "Impaired memory: inability to recall or retrieve bits of information or behavioral skills related to neurological damage from cerebral ischemia (NANDA-I 00131)",
     0.06),
    ("pre-B cell receptor complex",
     "Disturbed sleep pattern: time-limited disruption of sleep amount and quality related to hospitalization neurological symptoms and pain (NANDA-I 00198)",
     0.05),
    ("Secretory granule",
     "Caregiver role strain: difficulty in performing the family caregiver role related to complexity duration and demands of post-stroke care (NANDA-I 00061)",
     0.06),
    ("Extracellular region",
     "Impaired verbal communication: decreased ability to receive process transmit and use a system of symbols related to aphasia after stroke (NANDA-I 00051)",
     0.07),
    ("Cytoplasmic vesicle",
     "Impaired swallowing: abnormal functioning of swallowing mechanism compromising nutrition and airway protection due to neurological deficit (NANDA-I 00103)",
     0.08),
    ("Disulfide bond",
     "Impaired physical mobility: limitation in independent purposeful physical movement related to neurological impairment and hemiplegia (NANDA-I 00085)",
     0.03),
    ("Signal",
     "Powerlessness: lived experience of lack of control over a situation including a perception that one's actions do not significantly affect outcome related to stroke-induced dependency (NANDA-I 00125)",
     0.06),
    ("B cell receptor signaling pathway",
     "Fatigue: overwhelming sustained sense of exhaustion and decreased capacity for physical and mental work related to neurological disease (NANDA-I 00093)",
     0.08),
    ("Immunoreceptor tyrosine-based activation motif",
     "Constipation: decrease in normal bowel movement frequency and consistency related to immobility and neurological dysfunction (NANDA-I 00011)",
     0.04),
    ("Positive regulation of T cell activation",
     "Disturbed sleep pattern: time-limited disruption of sleep amount and quality related to hospitalization neurological symptoms and pain (NANDA-I 00198)",
     0.05),
]

lexical_hard_negatives = [
    # "infection" no nome da via, mas o diagnóstico correto
    # para esta via (imunossupressão B-cell) NÃO é infecção direta
    ("Unusual infection",
     "Risk for injury: susceptibility to physical damage due to "
     "sensory and motor deficit impaired balance and altered "
     "consciousness after stroke (NANDA-I 00035)",
     0.12),

    ("Unusual infection",
     "Risk for pressure injury: susceptibility to localized damage "
     "to skin and underlying tissue usually over bony prominence "
     "related to immobility and sensory deficit (NANDA-I 00249)",
     0.10),

    # "immune" no nome da via (Innate Immune System), mas diagnóstico
    # não-imune — o modelo NÃO deve dar score alto por "immune"
    ("Innate Immune System",
     "Impaired physical mobility: limitation in independent "
     "purposeful physical movement related to neurological "
     "impairment and hemiplegia (NANDA-I 00085)",
     0.06),

    ("Innate Immune System",
     "Disturbed sleep pattern: time-limited disruption of sleep "
     "amount and quality related to hospitalization neurological "
     "symptoms and pain (NANDA-I 00198)",
     0.05),

    # "Immune response" (via) ≠ "Immune" nos labels NANDA de forma
    # automática — a conexão deve ser mecanística
    ("Immune response",
     "Caregiver role strain: difficulty in performing the family "
     "caregiver role related to complexity duration and demands "
     "of post-stroke care (NANDA-I 00061)",
     0.04),

    ("Immune response",
     "Constipation: decrease in normal bowel movement frequency "
     "and consistency related to immobility and neurological "
     "dysfunction (NANDA-I 00011)",
     0.05),

    # "Immune effector process" não implica todos os diagnósticos
    # de segurança/proteção por força do token "immune"
    ("Immune effector process",
     "Risk for falls: susceptibility to falling that may cause "
     "physical harm and health deterioration related to "
     "neurological impairment and altered gait (NANDA-I 00155)",
     0.07),

    ("Immune effector process",
     "Bathing self-care deficit: impaired ability to independently "
     "perform bathing and hygiene activities related to hemiplegia "
     "and motor deficit (NANDA-I 00108)",
     0.05),

    # "Innate immunity" (via Keyword STRING) ≠ todos os diagnósticos
    # de infecção. Relevância moderada-baixa para diagnósticos
    # não-inatos (adaptativo é diferente de inato)
    ("Innate immunity",
     "Anxiety: vague uneasy feeling of discomfort or dread "
     "accompanied by autonomic response related to neurological "
     "deficit uncertainty and loss of independence (NANDA-I 00146)",
     0.09),

    ("Innate immunity",
     "Grieving: functional process that includes emotional physical "
     "spiritual social and intellectual responses related to loss "
     "of neurological function and previous lifestyle "
     "(NANDA-I 00136)",
     0.05),
]

# --- Expansão: ≥40 hard negatives (0.05–0.15) + ≥40 moderados (0.35–0.55) ---
IMMUNE_PW = [
    "Neutrophil degranulation", "Innate Immune System", "Adaptive immunity", "Innate immunity",
    "Immune response", "B cell activation", "Lymphocyte differentiation", "T cell differentiation",
    "Positive regulation of immune system process", "Antigen receptor-mediated signaling pathway",
    "Unusual infection", "Immunoglobulin production", "Regulation of immune response",
    "Immune effector process", "CD22 mediated BCR regulation", "Adaptive immune response",
    "Positive regulation of T cell activation", "Cell surface receptor signaling pathway",
    "B cell proliferation", "Secretory granule", "Extracellular region", "Cytoplasmic vesicle",
    "Lymphocyte activation", "B cell receptor signaling pathway", "Immunological synapse formation",
]
NANDA_NON_IMMUNE = [
    "Impaired verbal communication: decreased ability to receive process transmit and use a system of symbols related to aphasia after stroke (NANDA-I 00051)",
    "Constipation: decrease in normal bowel movement frequency and consistency related to immobility and neurological dysfunction (NANDA-I 00011)",
    "Disturbed sleep pattern: time-limited disruption of sleep amount and quality related to hospitalization neurological symptoms and pain (NANDA-I 00198)",
    "Caregiver role strain: difficulty in performing the family caregiver role related to complexity duration and demands of post-stroke care (NANDA-I 00061)",
    "Impaired physical mobility: limitation in independent purposeful physical movement related to neurological impairment and hemiplegia (NANDA-I 00085)",
    "Dressing self-care deficit: impaired ability to dress and undress independently related to motor and sensory deficit after stroke (NANDA-I 00109)",
    "Bathing self-care deficit: impaired ability to independently perform bathing and hygiene activities related to hemiplegia and motor deficit (NANDA-I 00108)",
    "Impaired memory: inability to recall or retrieve bits of information or behavioral skills related to neurological damage from cerebral ischemia (NANDA-I 00131)",
    "Hopelessness: subjective state in which individual sees limited or no alternatives or personal choices and is unable to mobilize energy related to neurological disability (NANDA-I 00124)",
    "Powerlessness: lived experience of lack of control over a situation including a perception that one's actions do not significantly affect outcome related to stroke-induced dependency (NANDA-I 00125)",
    "Impaired swallowing: abnormal functioning of swallowing mechanism compromising nutrition and airway protection due to neurological deficit (NANDA-I 00103)",
    "Interrupted family processes: change in family relationships and functioning related to acute illness and hospitalization of family member (NANDA-I 00060)",
    "Impaired urinary elimination: dysfunction of urinary elimination related to neurogenic bladder dysfunction after stroke (NANDA-I 00016)",
    "Disturbed body image: confusion in mental picture of one's physical self related to functional and physical loss after stroke and hemiplegia (NANDA-I 00118)",
    "Fear: response to perceived threat that is consciously recognized as a danger related to disability dependency and uncertain prognosis (NANDA-I 00148)",
    "Grieving: functional process that includes emotional physical spiritual social and intellectual responses related to loss of neurological function and previous lifestyle (NANDA-I 00136)",
    "Imbalanced nutrition less than body requirements: inadequate nutritional intake related to dysphagia and impaired swallowing after stroke (NANDA-I 00002)",
    "Risk for aspiration: susceptibility to entry of secretions into tracheobronchial passages due to impaired swallowing and reduced consciousness (NANDA-I 00039)",
    "Anxiety: vague uneasy feeling of discomfort or dread accompanied by autonomic response related to neurological deficit uncertainty and loss of independence (NANDA-I 00146)",
]
NANDA_SECOND_ORDER = [
    "Hyperthermia: core body temperature above normal diurnal range related to systemic inflammatory response and post-stroke infection (NANDA-I 00007)",
    "Impaired tissue integrity: damage to mucous membrane corneal integumentary or subcutaneous tissues related to ischemic injury and impaired circulation (NANDA-I 00044)",
    "Risk for injury: susceptibility to physical damage due to sensory and motor deficit impaired balance and altered consciousness after stroke (NANDA-I 00035)",
    "Acute confusion: abrupt onset of reversible disturbances of consciousness attention cognition and perception related to neuroinflammation and cerebral ischemia (NANDA-I 00128)",
    "Risk for pressure injury: susceptibility to localized damage to skin and underlying tissue usually over bony prominence related to immobility and sensory deficit (NANDA-I 00249)",
    "Decreased intracranial adaptive capacity: intracranial fluid dynamic mechanisms compromised resulting in repeated disproportionate increases in intracranial pressure (NANDA-I 00049)",
    "Risk for ineffective cerebral tissue perfusion: susceptibility to decrease in cerebral tissue circulation related to cerebral ischemia edema and blood-brain barrier disruption (NANDA-I 00201)",
    "Fatigue: overwhelming sustained sense of exhaustion and decreased capacity for physical and mental work related to neurological disease (NANDA-I 00093)",
    "Acute pain: unpleasant sensory and emotional experience associated with actual or potential tissue damage related to neurological injury and post-stroke pain (NANDA-I 00132)",
    "Impaired comfort: perceived lack of ease relief and transcendence in physical psychospiritual environmental and social dimensions related to neurological symptoms (NANDA-I 00214)",
]

extra_hard_negatives = [
    ("Neutrophil degranulation", NANDA_NON_IMMUNE[0], 0.05),
    ("Adaptive immunity", NANDA_NON_IMMUNE[1], 0.05),
    ("Innate Immune System", NANDA_NON_IMMUNE[2], 0.10),
    ("B cell activation", NANDA_NON_IMMUNE[3], 0.08),
    ("Lymphocyte differentiation", NANDA_NON_IMMUNE[4], 0.12),
]
seen_pairs = {(p, d) for p, d, _ in base_training_pairs}
for p, d, _ in lexical_hard_negatives:
    seen_pairs.add((p, d))
for t in extra_hard_negatives:
    seen_pairs.add((t[0], t[1]))
for pw in IMMUNE_PW:
    for nd in NANDA_NON_IMMUNE:
        if len(extra_hard_negatives) >= 45:
            break
        if (pw, nd) in seen_pairs:
            continue
        seen_pairs.add((pw, nd))
        lbl = 0.05 + (len(extra_hard_negatives) % 11) * 0.01
        extra_hard_negatives.append((pw, nd, round(min(lbl, 0.15), 2)))
    if len(extra_hard_negatives) >= 45:
        break

extra_moderate_pairs = [
    ("Innate Immune System", NANDA_SECOND_ORDER[0], 0.48),
    ("Neutrophil degranulation", NANDA_SECOND_ORDER[1], 0.55),
]
for t in extra_moderate_pairs:
    seen_pairs.add((t[0], t[1]))
for pw in IMMUNE_PW:
    for nd in NANDA_SECOND_ORDER:
        if len(extra_moderate_pairs) >= 45:
            break
        if (pw, nd) in seen_pairs:
            continue
        seen_pairs.add((pw, nd))
        lbl = 0.35 + (len(extra_moderate_pairs) % 21) * 0.01
        extra_moderate_pairs.append((pw, nd, round(min(lbl, 0.55), 2)))
    if len(extra_moderate_pairs) >= 45:
        break

extra_pairs = extra_hard_negatives + extra_moderate_pairs
training_pairs = base_training_pairs + extra_pairs + lexical_hard_negatives

print(f"✓ {len(training_pairs)} pares total")
print(f"  Base          : {len(base_training_pairs)}")
print(f"  Extra (expansão): {len(extra_pairs)}")
print(f"  Lexical hard neg: {len(lexical_hard_negatives)}")
alta_rel  = sum(1 for _,_,l in training_pairs if l >= 0.85)
moderada  = sum(1 for _,_,l in training_pairs if 0.35 <= l < 0.85)
baixa     = sum(1 for _,_,l in training_pairs if l < 0.35)
print(f"  Alta relevância (≥0.85) : {alta_rel}")
print(f"  Moderada (0.35–0.84)    : {moderada}")
print(f"  Baixa/hard (<0.35)      : {baixa}")


✓ 165 pares total
  Base          : 65
  Extra (expansão): 90
  Lexical hard neg: 10
  Alta relevância (≥0.85) : 22
  Moderada (0.35–0.84)    : 67
  Baixa/hard (<0.35)      : 76


In [2]:
# ============================================================
# CÉLULA 3 — Dataset para MNRL + split estratificado
# MultipleNegativesRankingLoss usa apenas pares positivos
# (alta relevância ≥ 0.70) no treino. Pares de baixa
# relevância ficam APENAS no conjunto de validação para
# medir capacidade de rejeição do modelo.
# ============================================================

from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ── Separar pares por faixa de label ─────────────────────
positive_pairs   = [(p, d, l) for p, d, l in training_pairs if l >= 0.70]
hard_neg_pairs   = [(p, d, l) for p, d, l in training_pairs if l <= 0.20]
moderate_pairs   = [(p, d, l) for p, d, l in training_pairs
                    if 0.20 < l < 0.70]

print(f"Pares positivos  (≥0.70, usados no treino MNRL): {len(positive_pairs)}")
print(f"Hard negatives   (≤0.20, usados só no val)     : {len(hard_neg_pairs)}")
print(f"Moderados        (0.20–0.70, usados só no val) : {len(moderate_pairs)}")

# ── Conjunto de validação: mistura TODOS os pares ────────
# O val avalia se o modelo discrimina bem TODA a faixa de
# relevância, incluindo hard negatives e moderados.
all_val_candidates = positive_pairs + hard_neg_pairs + moderate_pairs
val_examples = [
    InputExample(texts=[p, d], label=float(l))
    for p, d, l in all_val_candidates
]

# Split estratificado para val: garante representação de
# todas as faixas no subconjunto de avaliação
labels_arr    = np.array([ex.label for ex in val_examples])
labels_binned = pd.cut(
    labels_arr,
    bins=[-0.001, 0.25, 0.65, 1.001],
    labels=["low", "mid", "high"]
)
use_stratify = int(labels_binned.value_counts().min()) >= 2
_, val_ex = train_test_split(
    val_examples,
    test_size=0.25,
    stratify=labels_binned if use_stratify else None,
    random_state=42
)

# ── Treino MNRL: apenas pares positivos ──────────────────
# MNRL espera InputExample com texts=[sentence_a, sentence_b]
# SEM campo label — o positivo é o par; os negativos são
# os outros itens do mesmo batch (in-batch negatives).
train_examples = [
    InputExample(texts=[p, d])
    for p, d, _ in positive_pairs
]

class InputExampleDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

train_dataset = InputExampleDataset(train_examples)

# Batch size 16: cada batch terá 16 pares positivos e
# 15 negativos implícitos por âncora — suficiente para
# este tamanho de dataset.
train_dataloader = DataLoader(
    train_dataset, shuffle=True, batch_size=16
)

print(f"\n✓ Treino MNRL : {len(train_examples)} pares positivos")
print(f"  Validação   : {len(val_ex)} exemplos (todas as faixas)")
print(f"  Estratificação val: {'sim' if use_stratify else 'não'}")


Pares positivos  (≥0.70, usados no treino MNRL): 34
Hard negatives   (≤0.20, usados só no val)     : 76
Moderados        (0.20–0.70, usados só no val) : 55

✓ Treino MNRL : 34 pares positivos
  Validação   : 42 exemplos (todas as faixas)
  Estratificação val: sim


In [ ]:
# ============================================================
# CÉLULA 4 — Modelo base + MultipleNegativesRankingLoss
#
# Por que MNRL em vez de CosineSimilarityLoss:
# CosineSimilarityLoss otimiza valor absoluto de similaridade,
# empurrando todos os embeddings do domínio para uma região
# densa → colapso (ratio H/Hmax > 0.94 em 35/35 vias).
#
# MNRL otimiza ORDEM RELATIVA via contraste in-batch:
# dado par (via_i, diag_i), todos os outros diag_j do batch
# são negativos implícitos. Isso preserva discriminação fina
# sem comprimir a distribuição de scores.
# ============================================================

model = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT')

# MultipleNegativesRankingLoss — não requer label explícito,
# usa in-batch negatives automaticamente
train_loss = losses.MultipleNegativesRankingLoss(model)

print("✓ Modelo base carregado: pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT")
print("✓ Loss: MultipleNegativesRankingLoss (in-batch negatives)")
print(f"  Batch size: 16 → até 15 negativos implícitos por âncora")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Modelo base carregado: pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT
✓ Loss: MultipleNegativesRankingLoss (in-batch negatives)
  Batch size: 16 → até 15 negativos implícitos por âncora


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "c:\Users\Usuario\anaconda3\Lib\site-packages\httpx\_transports\default.py", line 101, in map_httpcore_exceptions
    yield
  File "c:\Users\Usuario\anaconda3\Lib\site-packages\httpx\_transports\default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "c:\Users\Usuario\anaconda3\Lib\site-packages\httpcore\_sync\connection_pool.py", line 256, in handle_request
    raise exc from None
  File "c:\Users\Usuario\anaconda3\Lib\site-packages\httpcore\_sync\connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
        pool_request.request
    )
  File "c:\Users\Usuario\anaconda3\Lib\site-packages\httpcore\_sync\connection.py", line 103, in handle_request
    return self._connection.handle_request(request)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "c:\Users\Usuario\anaconda3\Lib\site-packages\httpcore\_sync\http11.py", line 

In [4]:
# ============================================================
# CÉLULA 5 — Fine-tuning com validação durante o treino
# epochs=15, warmup_steps dinâmico para MNRL
# ============================================================

%pip install -q -U "accelerate>=1.1.0" "transformers[torch]"

import os
import gc
from datetime import datetime
import accelerate
import transformers
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

print(f"✓ accelerate: {accelerate.__version__}")
print(f"✓ transformers: {transformers.__version__}")

# Evita lock de arquivos no Windows (ex.: model_ft/model_base carregados do mesmo path)
for _name in ("model_ft", "model_base"):
    if _name in globals():
        del globals()[_name]
gc.collect()

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=[p.texts[0] for p in val_ex],
    sentences2=[p.texts[1] for p in val_ex],
    scores=[p.label for p in val_ex],
    name="nanda_val",
)
eval_steps = max(1, len(train_examples) // 2)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
train_output_path = f"outputs/sbert_nanda_finetuned_{run_id}"
os.makedirs(train_output_path, exist_ok=True)

# MNRL converge mais rápido que CosineSimilarityLoss
# porque o gradiente é mais informativo (múltiplos negativos
# por step). Mais épocas compensam o dataset menor de treino.
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=15,
    evaluation_steps=eval_steps,
    warmup_steps=max(10, len(train_examples) // 4),
    output_path=train_output_path,
    save_best_model=True,
    show_progress_bar=True,
)
print(f"✓ Fine-tuning concluído (melhor checkpoint conforme validação) em: {train_output_path}")


Note: you may need to restart the kernel to use updated packages.
✓ accelerate: 1.13.0
✓ transformers: 5.5.0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

c:\Users\Usuario\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Nanda Val Pearson Cosine,Nanda Val Spearman Cosine
3,No log,No log,0.534351,0.464609
6,No log,No log,0.494014,0.403765
9,No log,No log,0.433411,0.373099
12,No log,No log,0.524843,0.480591
15,No log,No log,0.620859,0.581512
17,No log,No log,0.657035,0.618749
18,No log,No log,0.670440,0.634000
21,No log,No log,0.686395,0.641707
24,No log,No log,0.689641,0.649252
27,No log,No log,0.691163,0.653633


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Fine-tuning concluído (melhor checkpoint conforme validação) em: outputs/sbert_nanda_finetuned_20260408_210212


In [5]:
# ============================================================
# CÉLULA 5b — Verificação e salvamento robusto do checkpoint
# ============================================================

import os, json
from pathlib import Path
from sentence_transformers import SentenceTransformer

SAVE_PATH = "outputs/sbert_nanda_finetuned"
os.makedirs(SAVE_PATH, exist_ok=True)

# ── Passo 1: verificar se checkpoint existente é carregável ──
checkpoint_ok = False
peso_safetensors = os.path.join(SAVE_PATH, "model.safetensors")
peso_pytorch     = os.path.join(SAVE_PATH, "pytorch_model.bin")
peso_existe      = os.path.exists(peso_safetensors) or                    os.path.exists(peso_pytorch)

if peso_existe:
    try:
        _test = SentenceTransformer(SAVE_PATH)
        _test.encode(["test"], show_progress_bar=False)
        tamanho = os.path.getsize(
            peso_safetensors if os.path.exists(peso_safetensors)
            else peso_pytorch
        )
        print(f"✓ Checkpoint existente carregável ({tamanho/1e6:.1f} MB)")
        checkpoint_ok = True
        del _test
    except Exception as e:
        print(f"⚠️  Checkpoint existente corrompido: {e}")
        print("   Procedendo com salvamento a partir do modelo em memória.")
else:
    print("⚠️  Nenhum checkpoint encontrado. Salvando modelo em memória.")

# ── Passo 2: salvar se necessário ────────────────────────────
if not checkpoint_ok:
    salvo = False

    # Estratégia 1: sentence-transformers padrão
    try:
        model.save(SAVE_PATH)
        _test = SentenceTransformer(SAVE_PATH)
        _test.encode(["test"], show_progress_bar=False)
        del _test
        print(f"✓ Estratégia 1 (save padrão): checkpoint salvo e verificado")
        salvo = True
    except Exception as e1:
        print(f"⚠️  Estratégia 1 falhou: {e1}")

    # Estratégia 2: pytorch state_dict
    if not salvo:
        try:
            import torch
            transformer_module = model[0]
            torch.save(
                transformer_module.auto_model.state_dict(),
                os.path.join(SAVE_PATH, "pytorch_model.bin")
            )
            transformer_module.auto_model.config.save_pretrained(SAVE_PATH)
            transformer_module.tokenizer.save_pretrained(SAVE_PATH)
            cfg = {"max_seq_length": transformer_module.max_seq_length,
                   "do_lower_case": False}
            with open(os.path.join(SAVE_PATH,
                      "sentence_bert_config.json"), "w") as f:
                json.dump(cfg, f)
            print("✓ Estratégia 2 (state_dict): checkpoint salvo")
            salvo = True
        except Exception as e2:
            print(f"⚠️  Estratégia 2 falhou: {e2}")

    # Estratégia 3: pickle de emergência
    if not salvo:
        import pickle
        pkl_path = "outputs/sbert_nanda_finetuned_emergency.pkl"
        with open(pkl_path, "wb") as f:
            pickle.dump(model, f)
        print(f"✓ Estratégia 3 (pickle): {pkl_path}")
        print("  ⚠️  Pickle não é recomendado para produção.")
        salvo = True

    assert salvo, "Nenhuma estratégia de salvamento funcionou."

# ── Passo 3: registrar metadados do checkpoint ────────────────
meta = {
    "checkpoint_verificado": checkpoint_ok,
    "save_path": SAVE_PATH,
    "arquivos": [f for f in os.listdir(SAVE_PATH)
                 if os.path.isfile(os.path.join(SAVE_PATH, f))],
    "spearman_honesto": 0.477,   # atualizar após cada re-treino
    "spearman_base": 0.444,
    "delta_spearman": 0.033,
    "modelo_base": "pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT",
    "loss": "MultipleNegativesRankingLoss",
    "epochs": 15,
    "n_train_pairs": 44,
}
with open(os.path.join(SAVE_PATH, "checkpoint_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print()
print("=== Conteúdo de outputs/sbert_nanda_finetuned/ ===")
for fname in sorted(os.listdir(SAVE_PATH)):
    fp = os.path.join(SAVE_PATH, fname)
    if os.path.isfile(fp):
        print(f"  {fname:45s}  {os.path.getsize(fp)/1e6:.2f} MB")


⚠️  Nenhum checkpoint encontrado. Salvando modelo em memória.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Estratégia 1 (save padrão): checkpoint salvo e verificado

=== Conteúdo de outputs/sbert_nanda_finetuned/ ===
  README.md                                      0.02 MB
  checkpoint_meta.json                           0.00 MB
  config.json                                    0.00 MB
  config_sentence_transformers.json              0.00 MB
  model.safetensors                              437.95 MB
  modules.json                                   0.00 MB
  sentence_bert_config.json                      0.00 MB
  tokenizer.json                                 0.71 MB
  tokenizer_config.json                          0.00 MB


In [6]:
# -- Cell 6: Métricas no conjunto de validação ------------------------------
# Spearman, MSE no val; comparação com o modelo BASE (sem fine-tuning).

# ============================================================
# CÉLULA 6 — Relatório de métricas (val_ex da Célula 3)
# ============================================================

import json
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error
from sentence_transformers import SentenceTransformer

BASE_NAME = "pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT"
FT_PATH = "outputs/sbert_nanda_finetuned"

s1 = [p.texts[0] for p in val_ex]
s2 = [p.texts[1] for p in val_ex]
y_true = np.array([p.label for p in val_ex], dtype=float)


def cosine_preds(model, a, b):
    e1 = model.encode(a, convert_to_numpy=True, show_progress_bar=False)
    e2 = model.encode(b, convert_to_numpy=True, show_progress_bar=False)
    e1 = e1 / np.linalg.norm(e1, axis=1, keepdims=True)
    e2 = e2 / np.linalg.norm(e2, axis=1, keepdims=True)
    return np.sum(e1 * e2, axis=1)


def load_sentence_transformer_compat(model_path: str) -> SentenceTransformer:
    try:
        return SentenceTransformer(model_path)
    except TypeError as e:
        if "unexpected keyword argument 'architecture'" not in str(e):
            raise

        cfg_path = os.path.join(model_path, "sentence_bert_config.json")
        if not os.path.exists(cfg_path):
            raise

        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)

        if isinstance(cfg, dict) and "architecture" in cfg:
            cfg.pop("architecture", None)
            with open(cfg_path, "w", encoding="utf-8") as f:
                json.dump(cfg, f, ensure_ascii=False, indent=2)

        return SentenceTransformer(model_path)


model_ft = load_sentence_transformer_compat(FT_PATH)
pred_ft = cosine_preds(model_ft, s1, s2)
sp_ft, _ = spearmanr(y_true, pred_ft)
mse_ft = mean_squared_error(y_true, pred_ft)

model_base = SentenceTransformer(BASE_NAME)
pred_base = cosine_preds(model_base, s1, s2)
sp_base, _ = spearmanr(y_true, pred_base)
mse_base = mean_squared_error(y_true, pred_base)

print("=== Validação (Spearman / MSE no mesmo val_ex) ===")
print(f"Spearman (fine-tuned):  {sp_ft:.4f}")
print(f"Spearman (base):        {sp_base:.4f}   ← sem fine-tuning")
print(f"MSE (fine-tuned):       {mse_ft:.5f}")
print(f"MSE (base):             {mse_base:.5f}")
print(f"Δ Spearman (ft - base): {sp_ft - sp_base:+.4f}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

HTTP Error 503 thrown while requesting HEAD https://huggingface.co/pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT/resolve/main/adapter_config.json
Retrying in 8s [Retry 5/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT/resolve/main/adapter_config.json


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "c:\Users\Usuario\anaconda3\Lib\site-packages\huggingface_hub\utils\_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\Usuario\anaconda3\Lib\site-packages\httpx\_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Server error '503 Service Unavailable' for url 'https://huggingface.co/api/models/pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT/commits/main'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/503



=== Validação (Spearman / MSE no mesmo val_ex) ===
Spearman (fine-tuned):  0.6544
Spearman (base):        0.4691   ← sem fine-tuning
MSE (fine-tuned):       0.06968
MSE (base):             0.14583
Δ Spearman (ft - base): +0.1853


## Integração com o notebook principal / Integration with main notebook

Após o fine-tuning concluir, o modelo salvo em `outputs/sbert_nanda_finetuned/` pode substituir o modelo base no pipeline principal.

After fine-tuning completes, the saved model in `outputs/sbert_nanda_finetuned/` can replace the base model in the main pipeline.

---

**Em `clinicalbert_vias_AVC.ipynb`, Célula 2**, substitua:

```python
model_sbert = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO-SCIFACT')
```

por:

```python
model_sbert = SentenceTransformer('outputs/sbert_nanda_finetuned')
```

Execute o notebook principal novamente **a partir da Célula 5** (geração de embeddings) para propagar o novo modelo pelo pipeline de similaridade, heatmaps e exportação.

Re-run the main notebook **starting from Cell 5** (embedding generation) to propagate the new model through the similarity, heatmap, and export pipeline.